# Learning Path Optimization System
### AI & ML Internship - Task 3

**Objective:** Using the OULAD dataset, this system analyzes student engagement and performance data to recommend personalized learning paths.

**Dataset:** studentInfo, studentVle (engagement clicks), studentAssessment - 32,593 students across 7 course modules.

## Readiness Score Formula

Each student's readiness score (0-100) is calculated based on 3 factors:
- Previous failed attempts (more attempts = lower score)
- Studied credits (penalty if below 60 credits)
- Engagement level (total clicks on the VLE platform)

## Learner Personas

Based on the readiness score, students are classified into 4 personas:
- **At-Risk Learner** (score < 60) - needs immediate support
- **Casual Explorer** (60-74) - moderate engagement
- **Steady Progressor** (75-89) - consistent performance  
- **High Achiever** (90+) - ready for advanced learning paths

## Key Insights - Dashboard

The dashboard presents 4 panels:
1. Persona Distribution - how many students fall into each category
2. Readiness Band Spread - overall readiness distribution
3. Engagement vs Persona correlation
4. Course Module Completion Rates

## Interactive Recommendation System

Enter a Student ID in the widget below to get a personalized learning path recommendation, based on their readiness score and persona.

In [21]:
import zipfile

# ZIP file-a extract pannurom
with zipfile.ZipFile('archive (2).zip', 'r') as zip_ref:
    zip_ref.extractall('.')

print("Extract aayiduchu!")

# Files list pannurom, verify pannurathukku
import os
print(os.listdir('.'))

Extract aayiduchu!
['.config', 'interactive_dashboard.html', 'studentInfo.csv', 'studentRegistration.csv', 'studentAssessment.csv', 'courses.csv', 'vle.csv', 'archive (2).zip', 'learning_model.pkl', 'studentVle.csv', 'assessments.csv', 'encoders.pkl', 'sample_data']


In [22]:
import pandas as pd

# Files load pannurom
studentInfo = pd.read_csv('studentInfo.csv')
courses = pd.read_csv('courses.csv')
assessments = pd.read_csv('assessments.csv')
studentAssessment = pd.read_csv('studentAssessment.csv')
studentRegistration = pd.read_csv('studentRegistration.csv')

print("Files load aayiduchu!")
print("Total rows (studentInfo):", len(studentInfo))

# Duplicate check
print("\nDuplicate rows:", studentInfo.duplicated().sum())

# Missing values check
print("\nMissing values before cleaning:")
print(studentInfo.isnull().sum())

# Missing values fill pannurom
studentInfo['imd_band'] = studentInfo['imd_band'].fillna('Unknown')

print("\nMissing values after cleaning:")
print(studentInfo.isnull().sum())

print("\nData cleaning complete!")

Files load aayiduchu!
Total rows (studentInfo): 32593

Duplicate rows: 0

Missing values before cleaning:
code_module                0
code_presentation          0
id_student                 0
gender                     0
region                     0
highest_education          0
imd_band                1111
age_band                   0
num_of_prev_attempts       0
studied_credits            0
disability                 0
final_result               0
dtype: int64

Missing values after cleaning:
code_module             0
code_presentation       0
id_student              0
gender                  0
region                  0
highest_education       0
imd_band                0
age_band                0
num_of_prev_attempts    0
studied_credits         0
disability              0
final_result            0
dtype: int64

Data cleaning complete!


In [23]:
# Readiness Score Calculate Pannura Function
def calculate_readiness_score(row):
    score = 100
    score -= row['num_of_prev_attempts'] * 15
    if row['studied_credits'] < 60:
        score -= 10
    score = max(score, 0)
    return score

studentInfo['readiness_score'] = studentInfo.apply(calculate_readiness_score, axis=1)

print("Readiness Score calculate aayiduchu!")
print(studentInfo[['id_student', 'num_of_prev_attempts', 'studied_credits', 'final_result', 'readiness_score']].head(10))

Readiness Score calculate aayiduchu!
   id_student  num_of_prev_attempts  studied_credits final_result  \
0       11391                     0              240         Pass   
1       28400                     0               60         Pass   
2       30268                     0               60    Withdrawn   
3       31604                     0               60         Pass   
4       32885                     0               60         Pass   
5       38053                     0               60         Pass   
6       45462                     0               60         Pass   
7       45642                     0              120         Pass   
8       52130                     0               90         Pass   
9       53025                     0               60         Pass   

   readiness_score  
0              100  
1              100  
2              100  
3              100  
4              100  
5              100  
6              100  
7              100  
8             

In [24]:
# Career Goals ku Required Courses
career_paths = {
    "Data Scientist": ["Statistics Foundations", "Python Programming", "Machine Learning Basics", "Feature Engineering", "Model Evaluation", "Capstone Project"],
    "ML Engineer": ["Statistics Foundations", "Python Programming", "Machine Learning Basics", "Feature Engineering", "Model Evaluation", "MLOps Fundamentals", "Capstone Project"],
    "Data Analyst": ["Statistics Foundations", "Excel & SQL", "Data Visualization", "Business Analytics", "Capstone Project"]
}

def get_learning_path(readiness_score, career_goal):
    all_courses = career_paths.get(career_goal, [])
    if readiness_score >= 85:
        remaining = all_courses[2:]
    elif readiness_score >= 60:
        remaining = all_courses[1:]
    else:
        remaining = all_courses
    return remaining

# Ella students ku um recommended path add pannurom
def get_recommended_courses_text(readiness_score, career_goal="Data Scientist"):
    courses = get_learning_path(readiness_score, career_goal)
    return " -> ".join(courses)

studentInfo['recommended_learning_path'] = studentInfo['readiness_score'].apply(get_recommended_courses_text)

print("Recommendation logic ready!")
studentInfo[['id_student', 'readiness_score', 'final_result', 'recommended_learning_path']].head(10)

Recommendation logic ready!


,id_student,readiness_score,final_result,recommended_learning_path
0,11391,100,Pass,Machine Learning Basics -> Feature Engineering...
1,28400,100,Pass,Machine Learning Basics -> Feature Engineering...
2,30268,100,Withdrawn,Machine Learning Basics -> Feature Engineering...
3,31604,100,Pass,Machine Learning Basics -> Feature Engineering...
4,32885,100,Pass,Machine Learning Basics -> Feature Engineering...
5,38053,100,Pass,Machine Learning Basics -> Feature Engineering...
6,45462,100,Pass,Machine Learning Basics -> Feature Engineering...
7,45642,100,Pass,Machine Learning Basics -> Feature Engineering...
8,52130,100,Pass,Machine Learning Basics -> Feature Engineering...
9,53025,100,Pass,Machine Learning Basics -> Feature Engineering...


In [25]:
studentVle = pd.read_csv('studentVle.csv')

engagement = studentVle.groupby('id_student')['sum_click'].sum().reset_index()
engagement.columns = ['id_student', 'total_clicks']

studentInfo = studentInfo.merge(engagement, on='id_student', how='left')
studentInfo['total_clicks'] = studentInfo['total_clicks'].fillna(0)

print("Engagement merged!")
print(studentInfo[['id_student', 'total_clicks']].head(10))

Engagement merged!
   id_student  total_clicks
0       11391         934.0
1       28400        1435.0
2       30268         281.0
3       31604        2158.0
4       32885        1034.0
5       38053        2445.0
6       45462        1492.0
7       45642        1428.0
8       52130        1894.0
9       53025        3158.0


In [26]:
def calculate_readiness_score(row):
    score = 100
    score -= row['num_of_prev_attempts'] * 15
    if row['studied_credits'] < 60:
        score -= 10
    if row['total_clicks'] < 100:
        score -= 30
    elif row['total_clicks'] < 500:
        score -= 15
    elif row['total_clicks'] < 1000:
        score -= 5
    score = max(score, 0)
    score = min(score, 100)
    return score

studentInfo['readiness_score'] = studentInfo.apply(calculate_readiness_score, axis=1)

print("Readiness Score updated!")
print(studentInfo[['id_student', 'total_clicks', 'readiness_score']].head(10))

Readiness Score updated!
   id_student  total_clicks  readiness_score
0       11391         934.0               95
1       28400        1435.0              100
2       30268         281.0               85
3       31604        2158.0              100
4       32885        1034.0              100
5       38053        2445.0              100
6       45462        1492.0              100
7       45642        1428.0              100
8       52130        1894.0              100
9       53025        3158.0              100


In [27]:
studentInfo['recommended_learning_path'] = studentInfo['readiness_score'].apply(get_recommended_courses_text)

print("Recommendation refreshed!")
studentInfo[['id_student', 'readiness_score', 'final_result', 'recommended_learning_path']].head(10)

Recommendation refreshed!


,id_student,readiness_score,final_result,recommended_learning_path
0,11391,95,Pass,Machine Learning Basics -> Feature Engineering...
1,28400,100,Pass,Machine Learning Basics -> Feature Engineering...
2,30268,85,Withdrawn,Machine Learning Basics -> Feature Engineering...
3,31604,100,Pass,Machine Learning Basics -> Feature Engineering...
4,32885,100,Pass,Machine Learning Basics -> Feature Engineering...
5,38053,100,Pass,Machine Learning Basics -> Feature Engineering...
6,45462,100,Pass,Machine Learning Basics -> Feature Engineering...
7,45642,100,Pass,Machine Learning Basics -> Feature Engineering...
8,52130,100,Pass,Machine Learning Basics -> Feature Engineering...
9,53025,100,Pass,Machine Learning Basics -> Feature Engineering...


In [28]:
# Score distribution paakalam
print(studentInfo['readiness_score'].describe())
print("\nScore ranges:")
print("85-100 (high):", len(studentInfo[studentInfo['readiness_score'] >= 85]))
print("60-84 (medium):", len(studentInfo[(studentInfo['readiness_score'] >= 60) & (studentInfo['readiness_score'] < 85)]))
print("Below 60 (low):", len(studentInfo[studentInfo['readiness_score'] < 60]))

# Vera vera path irukura example students kaatunga
print("\nDifferent path examples:")
print(studentInfo[studentInfo['readiness_score'] < 85][['id_student', 'readiness_score', 'recommended_learning_path']].head(5))

count    32593.000000
mean        86.788267
std         13.876633
min          0.000000
25%         75.000000
50%         90.000000
75%        100.000000
max        100.000000
Name: readiness_score, dtype: float64

Score ranges:
85-100 (high): 23516
60-84 (medium): 7893
Below 60 (low): 1184

Different path examples:
     id_student  readiness_score  \
44       135335               70   
118      281589               70   
125      292923               70   
136      305539               70   
165      344282               70   

                             recommended_learning_path  
44   Python Programming -> Machine Learning Basics ...  
118  Python Programming -> Machine Learning Basics ...  
125  Python Programming -> Machine Learning Basics ...  
136  Python Programming -> Machine Learning Basics ...  
165  Python Programming -> Machine Learning Basics ...  


In [29]:
import plotly.express as px

# Chart 1: Final Result Distribution
fig1 = px.histogram(studentInfo, x='final_result', color='final_result',
                     title='Final Result Distribution (Pass / Fail / Withdrawn / Distinction)')
fig1.show()

# Chart 2: Readiness Score by Final Result
fig2 = px.box(studentInfo, x='final_result', y='readiness_score', color='final_result',
              title='Readiness Score by Final Result')
fig2.show()

# Chart 3: Dropout Rate by Previous Attempts
dropout_data = studentInfo.groupby('num_of_prev_attempts')['final_result'].apply(
    lambda x: (x == 'Withdrawn').sum() / len(x) * 100
).reset_index(name='dropout_rate_percent')

fig3 = px.bar(dropout_data, x='num_of_prev_attempts', y='dropout_rate_percent',
              title='Dropout Rate % by Previous Attempts', text='dropout_rate_percent')
fig3.show()

# Chart 4: Readiness Score Distribution by Path Category
fig4 = px.histogram(studentInfo, x='readiness_score', color='final_result',
                     title='Readiness Score Distribution', nbins=20)
fig4.show()

print("Charts ready!")

Charts ready!


In [30]:
# ============================================
# STEP A: ML MODEL - FEATURE PREPARATION
# ============================================
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

# Model ku data ready pannurom
ml_data = studentInfo.copy()

# Target variable: student pass aaguva illa dropout/fail aaguva nu binary ah maathurom
# Pass/Distinction = 1 (Success), Fail/Withdrawn = 0 (At Risk)
ml_data['success'] = ml_data['final_result'].apply(
    lambda x: 1 if x in ['Pass', 'Distinction'] else 0
)

# Categorical columns-a numbers ah convert pannurom (ML models numbers mattum puriyum)
le_gender = LabelEncoder()
le_education = LabelEncoder()
le_disability = LabelEncoder()

ml_data['gender_encoded'] = le_gender.fit_transform(ml_data['gender'])
ml_data['education_encoded'] = le_education.fit_transform(ml_data['highest_education'])
ml_data['disability_encoded'] = le_disability.fit_transform(ml_data['disability'])

print("Data ready for ML model!")
print(ml_data[['id_student', 'success', 'gender_encoded', 'education_encoded']].head())

# ============================================
# STEP B: FEATURES SELECT PANNURATHU
# ============================================
features = [
    'num_of_prev_attempts',
    'studied_credits',
    'total_clicks',
    'gender_encoded',
    'education_encoded',
    'disability_encoded'
]

X = ml_data[features]
y = ml_data['success']

print("Features:", features)
print("Total samples:", len(X))
print("Success rate in data:", y.mean())

# ============================================
# STEP C: TRAIN-TEST SPLIT
# ============================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("\nTraining samples:", len(X_train))
print("Testing samples:", len(X_test))

# ============================================
# STEP D: MODEL TRAIN PANNURATHU (RANDOM FOREST)
# ============================================
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=8,
    random_state=42,
    class_weight='balanced'  # imbalanced data ku help pannum
)

model.fit(X_train, y_train)

print("\nModel training complete!")

# ============================================
# STEP E: MODEL EVALUATE PANNURATHU
# ============================================
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"\nModel Accuracy: {accuracy:.2%}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['At Risk', 'Success']))

# ============================================
# STEP F: FEATURE IMPORTANCE - EDHU MOST IMPORTANT NU PAAKALAM
# ============================================
importance_df = pd.DataFrame({
    'feature': features,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nFeature Importance (edhu factor romba impact pannuthu):")
print(importance_df)

# ============================================
# STEP G: ELLA STUDENTS-KUM ML-BASED READINESS SCORE GENERATE PANNURATHU
# ============================================
# Model-oda predict_proba, "success aagura probability"-a kudukum (0 to 1)
# Idha 0-100 score ah convert pannurom

all_predictions = model.predict_proba(X)[:, 1]  # success probability
ml_data['ml_readiness_score'] = (all_predictions * 100).round().astype(int)

print("\nML-based Readiness Score ready!")
print(ml_data[['id_student', 'final_result', 'num_of_prev_attempts', 'total_clicks', 'ml_readiness_score']].head(10))

# ============================================
# STEP H: OLD SCORE VS NEW ML SCORE COMPARE PANNURATHU
# ============================================
studentInfo['ml_readiness_score'] = ml_data['ml_readiness_score']

print("\nComparison - Rule-based vs ML-based Score:")
print(studentInfo[['id_student', 'readiness_score', 'ml_readiness_score', 'final_result']].head(10))

Data ready for ML model!
   id_student  success  gender_encoded  education_encoded
0       11391        1               1                  1
1       28400        1               0                  1
2       30268        0               0                  0
3       31604        1               0                  0
4       32885        1               0                  2
Features: ['num_of_prev_attempts', 'studied_credits', 'total_clicks', 'gender_encoded', 'education_encoded', 'disability_encoded']
Total samples: 32593
Success rate in data: 0.47203387230386895

Training samples: 26074
Testing samples: 6519

Model training complete!

Model Accuracy: 77.85%

Classification Report:
              precision    recall  f1-score   support

     At Risk       0.84      0.71      0.77      3442
     Success       0.73      0.85      0.78      3077

    accuracy                           0.78      6519
   macro avg       0.79      0.78      0.78      6519
weighted avg       0.79      0.78      0

In [31]:
# ============================================
# CONFUSION MATRIX - MODEL EPPADI PERFORM PANNUTHU NU VISUAL
# ============================================
from sklearn.metrics import confusion_matrix
import plotly.figure_factory as ff

cm = confusion_matrix(y_test, y_pred)

labels = ['At Risk', 'Success']

fig_cm = ff.create_annotated_heatmap(
    z=cm,
    x=labels,
    y=labels,
    colorscale='Blues',
    showscale=True
)
fig_cm.update_layout(
    title='Confusion Matrix - Model Prediction vs Actual',
    xaxis_title='Predicted',
    yaxis_title='Actual'
)
fig_cm.show()

print("Confusion Matrix:")
print(cm)
print("\nRow 0 = Actual At Risk, Row 1 = Actual Success")
print("Col 0 = Predicted At Risk, Col 1 = Predicted Success")

Confusion Matrix:
[[2447  995]
 [ 449 2628]]

Row 0 = Actual At Risk, Row 1 = Actual Success
Col 0 = Predicted At Risk, Col 1 = Predicted Success


In [32]:
import pickle

# Model-a save pannurom
with open('learning_model.pkl', 'wb') as f:
    pickle.dump(model, f)

# LabelEncoders-um save pannanum (dashboard la use pannanum)
with open('encoders.pkl', 'wb') as f:
    pickle.dump({
        'gender': le_gender,
        'education': le_education,
        'disability': le_disability
    }, f)

print("Model and encoders saved successfully!")

Model and encoders saved successfully!


In [33]:
# Persona and Readiness Band - CORRECT VERSION (idha overwrite pannும்)

def assign_persona(row):
    if row['readiness_score'] < 60:
        return 'At-Risk Learner'
    elif row['readiness_score'] < 75:
        return 'Casual Explorer'
    elif row['readiness_score'] < 90:
        return 'Steady Progressor'
    else:
        return 'High Achiever'

studentInfo['persona'] = studentInfo.apply(assign_persona, axis=1)

def assign_readiness_band(score):
    if score < 60:
        return 'Low'
    elif score < 75:
        return 'Medium'
    elif score < 90:
        return 'High'
    else:
        return 'Very High'

studentInfo['readiness_band'] = studentInfo['readiness_score'].apply(assign_readiness_band)

print("Persona and Readiness Band columns updated correctly!")
print(studentInfo[['id_student', 'readiness_score', 'persona', 'readiness_band']].head(10))

Persona and Readiness Band columns updated correctly!
   id_student  readiness_score            persona readiness_band
0       11391               95      High Achiever      Very High
1       28400              100      High Achiever      Very High
2       30268               85  Steady Progressor           High
3       31604              100      High Achiever      Very High
4       32885              100      High Achiever      Very High
5       38053              100      High Achiever      Very High
6       45462              100      High Achiever      Very High
7       45642              100      High Achiever      Very High
8       52130              100      High Achiever      Very High
9       53025              100      High Achiever      Very High


In [34]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 4-panel interactive dashboard
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Learner Personas Distribution', 'Readiness Band Distribution',
                     'Avg Engagement (Clicks) by Persona', 'Completion Rate % by Course Module'),
    specs=[[{'type': 'bar'}, {'type': 'pie'}],
           [{'type': 'bar'}, {'type': 'bar'}]]
)

# Panel 1: Persona Distribution
persona_order = ['At-Risk Learner', 'Casual Explorer', 'Steady Progressor', 'High Achiever']
persona_counts = studentInfo['persona'].value_counts().reindex(persona_order)
fig.add_trace(go.Bar(x=persona_counts.index, y=persona_counts.values, marker_color='#4C72B0', name='Personas'), row=1, col=1)

# Panel 2: Readiness Band Pie
band_counts = studentInfo['readiness_band'].value_counts()
fig.add_trace(go.Pie(labels=band_counts.index, values=band_counts.values, name='Band'), row=1, col=2)

# Panel 3: Avg Engagement by Persona
avg_engagement = studentInfo.groupby('persona')['total_clicks'].mean().reindex(persona_order)
fig.add_trace(go.Bar(x=avg_engagement.index, y=avg_engagement.values, marker_color='#DD8452', name='Engagement'), row=2, col=1)

# Panel 4: Completion Rate by Module
module_data = studentInfo.groupby('code_module')['final_result'].apply(
    lambda x: (x.isin(['Pass', 'Distinction']).sum() / len(x)) * 100
).sort_values(ascending=False)
fig.add_trace(go.Bar(x=module_data.index, y=module_data.values, marker_color='#55A868', name='Completion %'), row=2, col=2)

fig.update_layout(height=800, width=1000, title_text="Learning Path Optimization — Interactive Dashboard", showlegend=False)
fig.show()

# HTML file ah save pannunga - trainer ku share pannna easy
fig.write_html('interactive_dashboard.html')
print("Interactive dashboard saved as 'interactive_dashboard.html'!")

Interactive dashboard saved as 'interactive_dashboard.html'!


In [35]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Student ID search widget
student_id_input = widgets.Text(
    value='',
    placeholder='Student ID podunga (e.g. 11391)',
    description='Student ID:',
    style={'description_width': 'initial'}
)

output = widgets.Output()
search_button = widgets.Button(description='Recommendation Paar', button_style='primary')

def on_search_click(b):
    with output:
        clear_output()
        try:
            sid = int(student_id_input.value)
            student = studentInfo[studentInfo['id_student'] == sid]

            if student.empty:
                print(f"Student ID {sid} kandupidikala. Innoru ID try pannunga.")
            else:
                row = student.iloc[0]
                print(f"=== Student ID: {sid} ===")
                print(f"Readiness Score: {row['readiness_score']}")
                print(f"Persona: {row['persona']}")
                print(f"Readiness Band: {row['readiness_band']}")
                print(f"Total Engagement Clicks: {row['total_clicks']}")
                print(f"Previous Attempts: {row['num_of_prev_attempts']}")
                print(f"\nRecommended Learning Path:\n{row['recommended_learning_path']}")
        except ValueError:
            print("Sari illa, valid ah number podunga (e.g. 11391)")

search_button.on_click(on_search_click)

display(widgets.VBox([student_id_input, search_button, output]))

In [36]:
print("="*60)
print("LEARNING PATH OPTIMIZATION SYSTEM")
print("="*60)
print("Project Completed Successfully!")
print()

print("Features Implemented:")
print("✓ Data Preprocessing")
print("✓ Feature Engineering")
print("✓ Readiness Score Calculation")
print("✓ Machine Learning Prediction")
print("✓ Learner Persona Classification")
print("✓ Personalized Learning Path Recommendation")
print("✓ Interactive Plotly Dashboard")
print()

print("Status: PROJECT COMPLETED")
print("="*60)

LEARNING PATH OPTIMIZATION SYSTEM
Project Completed Successfully!

Features Implemented:
✓ Data Preprocessing
✓ Feature Engineering
✓ Readiness Score Calculation
✓ Machine Learning Prediction
✓ Learner Persona Classification
✓ Personalized Learning Path Recommendation
✓ Interactive Plotly Dashboard

Status: PROJECT COMPLETED
